# 🕸️ Graph Memory: Multi-Hop Reasoning over a Knowledge Graph

## Learning Objectives
In this notebook, you will learn:
1. **A graph memory store** - store facts as `(subject, relation, object)` edges with forward and backward indexes
2. **A memory writer** - extract triples from text with structured output and a small relation vocabulary
3. **A memory reader** - pick a start entity, traverse N hops, answer only from the traversal
4. **Multi-hop QA** - answer a question whose facts were never stated together
5. **A graph database** - load the same triples into Neo4j and run model-written Cypher read-only (optional)

## Where this fits
`01_episodic_with_semantic_memory.ipynb` recalled facts by similarity. Storing them as a graph keeps the *connections* between facts, so an agent can chain them.

## Prerequisites
- Structured output with Pydantic models (`llm.with_structured_output(...)`)
- Dependencies installed with `uv sync` at the repository root (includes `langchain-neo4j` for Part 5)
- `OPENAI_API_KEY` in the project-root `.env`
- Optional (Part 5): a Neo4j 5 instance with the APOC plugin, and `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD` in `.env`

| Property | Value |
|---|---|
| Origin | Knowledge graphs (Berners-Lee, semantic web); modern LLM revival: Microsoft GraphRAG (2024) |

## 📖 Definition
**Graph Memory** is an agent memory architecture that stores facts as a **knowledge graph** of `(subject, relation, object)` triples instead of flat text chunks or key-value pairs. Each fact becomes an edge connecting two entity nodes. Because facts are explicitly linked, the agent can **traverse** the graph outward from a starting entity to chain multiple facts together and answer **multi-hop questions** — questions whose answer requires combining two or more separate facts that were never stated together in any single sentence.

A flat memory store (a list of text snippets, or a vector index over them) retrieves whichever chunks are *textually similar* to the question. It has no notion of "chase the relation from A to B, then from B to C." Graph memory makes that traversal a first-class operation.

### High-Level Workflow

1. **Ingestion (Memory Writer):** Conversational or document text is passed to an LLM configured with `with_structured_output(...)` against a Pydantic schema. The LLM extracts every `(subject, relation, object)` triple it can find and the triples are inserted into the graph store.
2. **Storage:** Triples are stored as directed edges in an in-memory graph — `subject --relation--> object` — indexed by entity so both outgoing and incoming edges can be found quickly.
3. **Query (Memory Reader / QA):** Given a question, an LLM identifies the **starting entity** mentioned in the question. The graph store then performs a bounded **N-hop traversal** outward from that entity, collecting every triple reachable within `max_hops` hops.
4. **Answer Synthesis:** The collected triples (the traversal path) are formatted as context and handed to the LLM, which synthesizes the final answer — with the traversal path itself printed for inspection, so we can see exactly which facts were chained together.

### When to Use
* **Multi-hop question answering** — "Who manages the person who leads the team that shipped Project X?" style questions that require chaining relations.
* **Entity-centric long-term memory** — assistants that accumulate structured knowledge about people, organizations, and projects across many conversations or documents.
* **Knowledge-base construction** — turning unstructured notes/meeting transcripts into a queryable relationship graph over time.
* **Complement to episodic/semantic memory** — graph memory *is* one concrete implementation of the "semantic memory" half of an episodic+semantic stack (see `01_episodic_with_semantic_memory.ipynb`, which pairs a fact store with a vector store for episodic recall). This notebook goes deeper on the graph piece alone: extraction quality, explicit traversal, and multi-hop QA.

### Strengths & Weaknesses
* **Strengths:**
    * **Multi-hop reasoning:** Can answer questions a flat/unconnected retrieval store structurally cannot, because the connections between facts are preserved, not just the facts themselves.
    * **Explainability:** The exact path of triples used to answer a question can be printed/inspected — useful for debugging and trust.
    * **Deduplication & consolidation:** Facts about the same entity naturally consolidate around one node instead of being scattered across many similar-but-separate text chunks.
* **Weaknesses:**
    * **Extraction is lossy:** Triple extraction depends entirely on the LLM correctly identifying entities and relations; ambiguous coreference ("he", "the team") can silently corrupt the graph.
    * **Schema drift:** Without normalization, the same real-world relation can be written as `works_at`, `employed_by`, `works_for`, fragmenting the graph.
    * **No native fuzzy retrieval:** Unlike a vector store, a graph store needs the query's starting entity to be named (or resolved) correctly, or traversal never begins.
    * **Scale:** A naive in-memory adjacency structure (as used here) doesn't scale to millions of triples — a production system would move to a real graph database — Part 5 loads the same triples into Neo4j.

---
## 📦 Step 1: Environment Setup

The setup cell loads the API keys from the project-root `.env` and silences library deprecation warnings.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: extracts triples, picks the start entity and answers
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🕸️ Part 1: The Graph Memory Store

We represent the graph as a dict-of-dicts adjacency structure rather than pulling in `networkx` (not currently a dependency of this repo — see `pyproject.toml`). Each node stores its **outgoing** edges (`subject -> [(relation, object), ...]`) and we separately index **incoming** edges so traversal can also follow relations backwards (e.g. "who reports to X" as well as "who does X report to").

* `add_triple(subject, relation, object)` — inserts a directed edge `subject --relation--> object`. A fact extracted twice is stored once, and entity names match case-insensitively and without a leading "the".
* `query(entity, max_hops=2)` — a breadth-first traversal outward (in both directions) from `entity`, returning every triple reached within `max_hops` hops, along with the hop distance at which it was discovered. Every edge can be reached from both of its ends, so a visited-edge set reports each edge once.

In [ ]:
# ============================================================================
# PART 1: AN IN-MEMORY GRAPH STORE
# ============================================================================
from dataclasses import dataclass, field


@dataclass
class Triple:
    subject: str
    relation: str
    object: str

    def __str__(self) -> str:
        return f"({self.subject}) --[{self.relation}]--> ({self.object})"


class GraphMemoryStore:
    """A minimal in-memory knowledge graph of (subject, relation, object) triples.

    Stored as a dict-of-dicts adjacency structure:
        _out[entity] = [(relation, other_entity), ...]   outgoing edges
        _in[entity]  = [(relation, other_entity), ...]   incoming edges (for backward traversal)
    """

    def __init__(self) -> None:
        self._out: dict[str, list[tuple[str, str]]] = {}
        self._in: dict[str, list[tuple[str, str]]] = {}
        self._triples: list[Triple] = []
        self._edges: set[tuple[str, str, str]] = set()  # normalised (subject, relation, object)

    @staticmethod
    def _norm(entity: str) -> str:
        return entity.strip().lower().removeprefix("the ")  # "the Atlas team" is the "Atlas team" node

    def add_triple(self, subject: str, relation: str, object: str) -> Triple:
        s, o = self._norm(subject), self._norm(object)
        triple = Triple(subject=subject.strip(), relation=relation.strip(), object=object.strip())
        if (s, triple.relation.lower(), o) in self._edges:  # the same fact extracted twice: keep one edge
            return triple
        self._edges.add((s, triple.relation.lower(), o))
        self._out.setdefault(s, []).append((relation.strip(), o))
        self._in.setdefault(o, []).append((relation.strip(), s))
        # Make sure both endpoints exist as nodes even with no edges in one direction
        self._out.setdefault(o, [])
        self._in.setdefault(s, [])
        self._triples.append(triple)
        return triple

    def _display_name(self, normalized_entity: str) -> str:
        """Best-effort recovery of the original-cased entity name for display."""
        for t in self._triples:
            if self._norm(t.subject) == normalized_entity:
                return t.subject
            if self._norm(t.object) == normalized_entity:
                return t.object
        return normalized_entity

    def query(self, entity: str, max_hops: int = 2) -> list[dict]:
        """Breadth-first traversal outward (both directions) from `entity`.

        Returns a list of dicts: {"hop": int, "triple": Triple, "direction": "out"|"in"}
        capturing every triple discovered within `max_hops` hops of the start entity.
        """
        start = self._norm(entity)
        if start not in self._out and start not in self._in:
            return []

        visited_entities = {start}
        seen_edges = set()  # an edge is reachable from both of its ends: report it once
        frontier = [start]
        results: list[dict] = []

        for hop in range(1, max_hops + 1):
            next_frontier = []
            for node in frontier:
                for relation, other in self._out.get(node, []):
                    if (node, relation, other) in seen_edges:
                        continue
                    seen_edges.add((node, relation, other))
                    results.append(
                        {
                            "hop": hop,
                            "direction": "out",
                            "triple": Triple(self._display_name(node), relation, self._display_name(other)),
                        }
                    )
                    if other not in visited_entities:
                        visited_entities.add(other)
                        next_frontier.append(other)
                for relation, other in self._in.get(node, []):
                    if (other, relation, node) in seen_edges:
                        continue
                    seen_edges.add((other, relation, node))
                    results.append(
                        {
                            "hop": hop,
                            "direction": "in",
                            "triple": Triple(self._display_name(other), relation, self._display_name(node)),
                        }
                    )
                    if other not in visited_entities:
                        visited_entities.add(other)
                        next_frontier.append(other)
            frontier = next_frontier
            if not frontier:
                break

        return results

    def all_triples(self) -> list[Triple]:
        return list(self._triples)


graph_store = GraphMemoryStore()
print("Graph memory store initialized.")

---
## ✍️ Part 2: The Memory Writer

The writer takes a chunk of free text and asks the LLM (via `with_structured_output`) to extract every
`(subject, relation, object)` triple it contains. Two rules keep the graph consistent:

- **A small relation vocabulary** (`leads`, `reports_to`, `built`, `based_in`, …), so one real-world relation
  doesn't end up as `is`, `has` and `holds_role` on different edges.
- **The subject is whoever performs the action.** Passive sentences are turned around: "Project Helios was built
  by the Atlas team" becomes `(Atlas team, built, Project Helios)`. Without this rule the model tends to keep the
  sentence order and write `(Project Helios, built, Atlas team)` — an edge that says the project built the team.

In [ ]:
# ============================================================================
# PART 2: EXTRACT TRIPLES WITH A RELATION VOCABULARY
# ============================================================================
from pydantic import BaseModel, Field

RELATIONS = ["leads", "manages", "reports_to", "member_of", "built", "works_on", "holds_role", "based_in"]


class ExtractedTriple(BaseModel):
    subject: str = Field(description="Who or what performs the action: a person, team, project or org name.")
    relation: str = Field(description=f"A short, lowercase, snake_case verb. Prefer one of: {', '.join(RELATIONS)}.")
    object: str = Field(description="The entity or value the action applies to.")


class ExtractedTriples(BaseModel):
    """All (subject, relation, object) facts found in a piece of text."""

    triples: list[ExtractedTriple] = Field(default_factory=list)


extractor_llm = llm.with_structured_output(ExtractedTriples)


def memory_writer(text: str, store: GraphMemoryStore) -> list[Triple]:
    """Extract triples from `text` with the LLM and insert them into `store`."""
    prompt = (
        "Extract every factual (subject, relation, object) triple stated in the text below.\n"
        f"- Relations: short, lowercase, snake_case verbs. Use one of {RELATIONS} whenever it fits; "
        "invent a new one only when none does.\n"
        "- The subject is whoever performs the action. Turn passive sentences around: "
        "'The report was written by Ana' -> (Ana, wrote, The report).\n"
        "- Use the exact entity names as they appear in the text (people, teams, projects, orgs).\n\n"
        f"Text:\n{text}"
    )
    extracted = extractor_llm.invoke(prompt)
    return [store.add_triple(t.subject, t.relation, t.object) for t in extracted.triples]


print(f"✅ Memory writer ready (relation vocabulary: {RELATIONS})")

---
## 🔎 Part 3: The Memory Reader

Answering a multi-hop question happens in three steps:

1. **Identify the starting entity** — ask the LLM which entity in the graph the question is "about" (its jumping-off point for traversal).
2. **Traverse the graph** — call `store.query(entity, max_hops=...)` to collect every triple within N hops.
3. **Synthesize the answer** — hand the LLM only the retrieved triples (not the whole graph, and not the original source text) and ask it to answer using just that context, so we can verify the answer is actually coming from the traversal.

In [ ]:
# ============================================================================
# PART 3: START ENTITY -> N-HOP TRAVERSAL -> ANSWER
# ============================================================================
class StartEntity(BaseModel):
    entity: str = Field(description="The single entity name in the graph that the question is primarily about.")


entity_picker_llm = llm.with_structured_output(StartEntity)


def memory_reader(question: str, store: GraphMemoryStore, max_hops: int = 2) -> dict:
    """Answer `question` by traversing `store` outward from an LLM-identified start entity."""
    known_entities = sorted(store._out.keys())
    pick_prompt = (
        "Given the question below and this list of known entities in a knowledge graph, "
        "return the single entity name (copied exactly from the list) that the question's "
        "traversal should start from: the entity the question names explicitly, not one you infer.\n\n"
        f"Known entities: {known_entities}\n\n"
        f"Question: {question}"
    )
    start = entity_picker_llm.invoke(pick_prompt).entity

    traversal = store.query(start, max_hops=max_hops)
    context_lines = [f"hop {r['hop']}: {r['triple']}" for r in traversal]
    context = "\n".join(context_lines) if context_lines else "(no triples found)"

    answer_prompt = (
        "Answer the question using ONLY the facts listed below. Chain multiple facts together "
        "if needed to reach the answer. If the facts are insufficient, say so explicitly.\n\n"
        f"Facts (graph traversal from '{start}'):\n{context}\n\n"
        f"Question: {question}"
    )
    answer = llm.invoke(answer_prompt).content

    return {"start_entity": start, "traversal": traversal, "answer": answer}


print("✅ Memory reader ready")

---
## 🧪 Part 4: Multi-Hop Question Answering

We feed in facts about a fictional org across **separate** sentences/documents — no single sentence contains the full chain needed to answer our test question. A flat memory store retrieving by text similarity to the question would likely surface only the sentence that superficially matches the question's wording (about "Project Helios"), and miss the sentence about who manages its team lead. Graph memory instead chains the two facts together via traversal.

In [ ]:
# ============================================================================
# PART 4: INGEST FIVE SEPARATE SENTENCES
# ============================================================================
documents = [
    "Project Helios was built by the Atlas team.",
    "Priya Chandran leads the Atlas team.",
    "Priya Chandran reports to Marcus Webb, the VP of Engineering.",
    "Marcus Webb has been VP of Engineering since 2021.",
    "The Atlas team is based in the Bangalore office.",
]

for doc in documents:
    inserted = memory_writer(doc, graph_store)
    print(f"Ingested: {doc!r}")
    for t in inserted:
        print(f"   -> {t}")

In [ ]:
# ============================================================================
# THE GRAPH, AND THE THREE EDGES THE QUESTION NEEDS
# ============================================================================
print("All triples currently in the graph:")
for t in graph_store.all_triples():
    print(" ", t)

# "Who manages the person who leads the team that built Project Helios?" needs these edges, pointing this way
EXPECTED_CHAIN = [
    ("atlas team", "built", "project helios"),
    ("priya chandran", "leads", "atlas team"),
    ("priya chandran", "reports_to", "marcus webb"),
]
print()
for edge in EXPECTED_CHAIN:
    print(f"{'✅' if edge in graph_store._edges else '❌'} ({edge[0]}) --[{edge[1]}]--> ({edge[2]})")

### Discussion of the Output

Each sentence produced one or more isolated triples. The passive sentence came out active —
`(Atlas team) --[built]--> (Project Helios)` — and the checks above confirm the graph holds all three edges of
the chain, pointing the right way. No single document mentions Project Helios and Marcus Webb together, so a flat
text-similarity retriever asked "who manages the person who leads the team that built Project Helios?" has no
single chunk that contains the answer. Graph memory instead lets us **traverse**:
Project Helios ← (built) ← Atlas team ← (leads) ← Priya Chandran → (reports_to) → Marcus Webb.

In [ ]:
# ============================================================================
# ASK THE MULTI-HOP QUESTION
# ============================================================================
multi_hop_question = (
    "Who manages the person who leads the team that built Project Helios?"
)

result = memory_reader(multi_hop_question, graph_store, max_hops=3)

print(f"Start entity identified by LLM: {result['start_entity']!r}\n")
print("Traversal path (triples used as context):")
for r in result["traversal"]:
    print(f"  hop {r['hop']} [{r['direction']}]  {r['triple']}")

print("\nQuestion:", multi_hop_question)
print("Answer:  ", result["answer"])


def edges_in(traversal: list[dict]) -> set:
    """The normalised (subject, relation, object) edges of a traversal."""
    norm = GraphMemoryStore._norm
    return {(norm(r["triple"].subject), r["triple"].relation.lower(), norm(r["triple"].object)) for r in traversal}


print(f"\n{'✅' if set(EXPECTED_CHAIN) <= edges_in(result['traversal']) else '❌'} the traversal reached all three chain edges")
print(f"{'✅' if 'marcus webb' in result['answer'].lower() else '❌'} the answer names Marcus Webb")

### 4.1 🔁 The Same Question With One Hop

Lower `max_hops` to `1` and the traversal stops next to the start entity. The edges that lead to the manager are
two hops further out, so the reader should say the facts are insufficient instead of guessing.

In [ ]:
# ============================================================================
# THE SAME QUESTION WITH ONE HOP
# ============================================================================
short = memory_reader(multi_hop_question, graph_store, max_hops=1)

print(f"Start entity identified by LLM: {short['start_entity']!r}\n")
for r in short["traversal"]:
    print(f"  hop {r['hop']} [{r['direction']}]  {r['triple']}")
print("\nAnswer:  ", short["answer"])

reached_marcus = any("marcus webb" in (s, o) for s, _, o in edges_in(short["traversal"]))
print(f"\n{'✅' if not reached_marcus else '❌'} one hop never reaches Marcus Webb")

### Discussion of the Output

With three hops the traversal starts from `Project Helios`, reaches `Atlas team` through an incoming `built`
edge, then `Priya Chandran` through an incoming `leads` edge (found through the incoming-edge index), then
`Marcus Webb` through `reports_to` — three separate facts chained, each edge listed once. The printed path makes
the reasoning **inspectable**: we can see exactly which edges were walked to reach the answer, rather than trusting
an opaque retrieval score. With one hop the path ends at the team, Marcus Webb is never reached, and the reader
has to admit the facts are insufficient: multi-hop traversal, not single-hop lookup, is what answers the question.

---
## 🐘 Part 5: The Same Graph in Neo4j, Queried with Text-to-Cypher (Optional)

A graph database replaces the hand-written traversal with a query language. Here the triples extracted above
are loaded into Neo4j as `GraphDocument`s, and the model translates a question into **Cypher** using the
graph's schema, runs it read-only, and phrases the result. This part runs only when `NEO4J_URI`, `NEO4J_USERNAME` and
`NEO4J_PASSWORD` are set and the instance answers; otherwise each cell prints why and skips.

`langchain-neo4j` reads the graph schema through the **APOC** plugin (`apoc.meta.data`), so the instance needs
it: AuraDB ships APOC Core, and the Docker command below enables it with `NEO4J_PLUGINS`.

> ⚠️ The cell **clears the target database** (`MATCH (n) DETACH DELETE n`) — point it at a scratch instance.

### Set up a Neo4j instance

Pick **one** option. Both follow the official Neo4j docs.

**Option A: Neo4j AuraDB Free (cloud, no install)**. See [Creating an instance](https://neo4j.com/docs/aura/getting-started/create-instance/).

1. Sign in to the [Aura console](https://console.neo4j.io) and create an instance.
2. Choose the **AuraDB Free** tier. Each account can have only one Free instance.
3. Copy or download the generated credentials **now**. Aura does not show the password again.
4. Wait until the instance status shows **Running**.
5. Copy the URI, username and password from the credentials file into `.env`. The URI looks like `neo4j+s://<instance-id>.databases.neo4j.io`.

**Option B: Local Docker**. See [Neo4j with Docker](https://neo4j.com/docs/operations-manual/current/docker/introduction/).

```bash
docker run -d --name neo4j-scratch \
    --publish=7474:7474 --publish=7687:7687 \
    --env NEO4J_AUTH=neo4j/password123 \
    --env NEO4J_PLUGINS='["apoc"]' \
    neo4j:latest
```

- Port `7474` serves Neo4j Browser at http://localhost:7474. Port `7687` serves Bolt, which the driver uses.
- The username is always `neo4j`. The password must have at least 8 characters.
- Stop and remove the container later with `docker rm -f neo4j-scratch`.

Then set these variables in the project-root `.env` and restart the kernel:

```bash
NEO4J_URI=bolt://localhost:7687        # Aura: neo4j+s://<instance-id>.databases.neo4j.io
NEO4J_USERNAME=neo4j                   # Aura: the username from the credentials file
NEO4J_PASSWORD=password123             # Aura: the generated password
```

In [ ]:
# ============================================================================
# PART 5: LOAD THE TRIPLES INTO NEO4J (OPTIONAL)
# ============================================================================
import os
import re

from langchain_core.documents import Document
from langchain_neo4j import Neo4jGraph
from langchain_neo4j.graphs.graph_document import GraphDocument, Node, Relationship


def rel_type(relation: str) -> str:
    """A Neo4j relationship type: 'reports_to' -> 'REPORTS_TO'."""
    return re.sub(r"[^A-Za-z0-9]+", "_", relation).upper()


# One Neo4j node per normalised entity name: "The Atlas team" and "Atlas team" are the same node
norm = GraphMemoryStore._norm
node_ids = {}
for t in graph_store.all_triples():
    for name in (t.subject, t.object):
        node_ids.setdefault(norm(name), name)

neo4j_graph = None
if not all(os.getenv(k) for k in ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD")):
    print("⏭️  NEO4J_* not set — skipping Part 5. Start Neo4j with the Docker command above.")
else:
    try:
        neo4j_graph = Neo4jGraph()  # reads NEO4J_URI / NEO4J_USERNAME / NEO4J_PASSWORD; needs APOC for the schema
    except Exception as error:  # unreachable, wrong credentials or no APOC: Part 5 is optional
        print(f"⏭️  Neo4j not usable — skipping Part 5: {error}")

if neo4j_graph is not None:
    neo4j_graph.query("MATCH (n) DETACH DELETE n")  # scratch database: start clean

    # Convert our triples into a GraphDocument: nodes + typed relationships
    nodes = {key: Node(id=name, type="Entity") for key, name in node_ids.items()}
    relationships = [Relationship(source=nodes[norm(t.subject)], target=nodes[norm(t.object)], type=rel_type(t.relation))
                     for t in graph_store.all_triples()]
    neo4j_graph.add_graph_documents([GraphDocument(
        nodes=list(nodes.values()), relationships=relationships, source=Document(page_content="\n".join(documents)),
    )])
    neo4j_graph.refresh_schema()
    print(f"Loaded {len(nodes)} nodes and {len(relationships)} relationships.\n\nSchema:\n{neo4j_graph.schema}")

### 5.1 🔒 Run Model-Written Cypher Read-Only

The next step executes Cypher that the **model** wrote. That is untrusted code with full access to the database:
a confused or prompt-injected model can write `DETACH DELETE` or `SET` as easily as `MATCH`. LangChain's own
`GraphCypherQAChain` refuses to run until you pass `allow_dangerous_requests=True`, and its docs tell you to give
the database user the narrowest permissions possible. This notebook adds two checks of its own:

1. **Ask the planner first.** `EXPLAIN <query>` plans the query without running it, and the result summary says
   whether it only reads (`'r'`), writes (`'w'`, `'rw'`) or changes the schema (`'s'`). Only `'r'` is executed.
2. **Run it in a read transaction** (`routing_=RoutingControl.READ`), on a separate driver used only for this.

In production, also connect as a database user with a read-only role: that is the one control the server
enforces whatever the query says. The cell below shows a write attempt being refused before it runs.

In [ ]:
# ============================================================================
# PART 5.1: A READ-ONLY CLIENT FOR MODEL-WRITTEN CYPHER
# ============================================================================
from neo4j import GraphDatabase, RoutingControl

NEO4J_DB = os.getenv("NEO4J_DATABASE", "neo4j")


def run_read_only(cypher: str) -> list[dict]:
    """Run untrusted Cypher only if Neo4j's planner classifies it as read-only, inside a read transaction."""
    plan = reader.execute_query(f"EXPLAIN {cypher}", routing_=RoutingControl.READ, database_=NEO4J_DB)
    if plan.summary.query_type != "r":  # 'r' read · 'rw' read/write · 'w' write · 's' schema
        raise PermissionError(f"refused: Neo4j classifies this query as {plan.summary.query_type!r}, not read-only 'r'")
    records = reader.execute_query(cypher, routing_=RoutingControl.READ, database_=NEO4J_DB).records
    return [record.data() for record in records]


def node_count() -> int:
    return run_read_only("MATCH (n) RETURN count(n) AS n")[0]["n"]


if neo4j_graph is None:
    print("⏭️  Skipped: no Neo4j connection (see Part 5).")
else:
    reader = GraphDatabase.driver(os.environ["NEO4J_URI"],
                                  auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]))
    before = node_count()
    try:
        run_read_only("MATCH (n) DETACH DELETE n")  # what a confused or prompt-injected model might write
    except PermissionError as error:
        print(f"Write attempt: {error}")
    after = node_count()
    print(f"{'✅' if before == after else '❌'} nodes before/after the write attempt: {before}/{after}")

### 5.2 🧭 Text-to-Cypher With Direction Hints

A schema lists the relationship types, but the model still has to guess which way each edge points — and a wrong
guess returns no rows rather than an error. So the prompt shows one real edge per relationship type, the exact
entity ids, and a worked multi-hop example from a different graph.

In [ ]:
# ============================================================================
# PART 5.2: TEXT-TO-CYPHER WITH DIRECTION HINTS
# ============================================================================
CYPHER_PROMPT = """Convert the question into ONE read-only Cypher query for this Neo4j graph.
Every entity is an (:Entity {{id: ...}}) node. Use only these relationship types, in the direction shown
by the example edges.

Example edges (one per relationship type):
{examples}

Entity ids: {entities}

Worked example (a different graph, with edges (:Entity {{id: 'Ana'}})-[:WROTE]->(:Entity {{id: 'Report X'}})
and (:Entity {{id: 'Ana'}})-[:REPORTS_TO]->(:Entity {{id: 'Ben'}})):
Question: Who supervises the author of Report X?
Cypher: MATCH (author:Entity)-[:WROTE]->(:Entity {{id: 'Report X'}}), (author)-[:REPORTS_TO]->(boss:Entity) RETURN boss.id

Return only the query.

Question: {question}"""

# One real edge per relationship type shows the model which way each edge points
first_edge = {}
for t in graph_store.all_triples():
    first_edge.setdefault(rel_type(t.relation), t)
EXAMPLES = "\n".join(f"(:Entity {{id: '{node_ids[norm(t.subject)]}'}})-[:{rel}]->(:Entity {{id: '{node_ids[norm(t.object)]}'}})"
                     for rel, t in first_edge.items())
ENTITIES = sorted(node_ids.values())


def ask_neo4j(question: str) -> list[dict]:
    prompt = CYPHER_PROMPT.format(examples=EXAMPLES, entities=ENTITIES, question=question)
    cypher = re.sub(r"^```(cypher)?|```$", "", llm.invoke(prompt).content.strip(), flags=re.IGNORECASE).strip()
    print(f"\nQuestion: {question}\nCypher:   {cypher}")
    try:
        rows = run_read_only(cypher)
    except Exception as error:  # refused as a write, or invalid Cypher
        print(f"Query failed: {error}")
        return []
    print(f"Rows:     {rows}")
    print("Answer:  ", llm.invoke(f"Answer the question using only these query results.\n"
                                  f"Question: {question}\nResults: {rows}").content)
    return rows


print(f"Example edges:\n{EXAMPLES}")
if neo4j_graph is None:
    print("\n⏭️  Skipped: no Neo4j connection (see Part 5).")
else:
    ask_neo4j("Who leads the Atlas team?")
    rows = ask_neo4j(multi_hop_question)
    print(f"\n{'✅' if 'marcus webb' in str(rows).lower() else '❌'} the multi-hop query returned Marcus Webb")

### Discussion of the Output

The example edges printed above are the direction hints the model sees. When Part 5 runs, the single-hop
question needs one `LEADS` edge, and the multi-hop question has to walk `BUILT` and `LEADS` backwards from Project
Helios and `REPORTS_TO` forwards from the team lead; the last line checks in code that Marcus Webb came back.
Without the example edges the model tends to point an edge the wrong way, and the query returns `[]` instead of
an error. When Neo4j is not configured or can't be reached, each Part 5 cell prints why it skipped and the rest of
the notebook is unaffected.

---
## 📝 Summary

In this notebook, we learned:

### 1. A graph memory store
- Graph memory stores facts as `(subject, relation, object)` edges, so the *connections* between facts are a first-class, queryable structure; each fact is stored once and each edge is reported once per traversal

### 2. A memory writer
- `llm.with_structured_output(...)` turns text into triples; a small relation vocabulary and an active-voice rule keep edges consistent and pointing the right way

### 3. A memory reader
- Identify a start entity, traverse a bounded number of hops, and answer only from the triples found — with the path printed for inspection

### 4. Multi-hop QA
- Three hops chained three separately stated facts; one hop could not reach the answer

### 5. A graph database
- Neo4j replaces the hand-written traversal with Cypher; model-written Cypher runs only after `EXPLAIN` classifies it as read-only, inside a read transaction

**Production caveat:** the in-memory dict-of-dicts store here teaches the mechanism. A real system uses a graph database, entity resolution (`works_at` vs `employed_by`, "Priya" vs "Priya Chandran") and pruning as the graph grows.

### Next Steps
- **`03_memgpt_tiered_memory.ipynb`**: an agent that pages facts between a small working context and archival storage